In [2]:
# =====================================================================
# LAB 1: NLU BANCÁRIO COM REGRESSÃO LOGÍSTICA
# =====================================================================
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
dados_treino = [
    ("Quero fazer uma transferencia de 500 reais", "transferencia"),
    ("Preciso transferir 200 reais para outra conta", "transferencia"),
    ("Enviar dinheiro para conta corrente 1000 reais", "transferencia"),
    ("Qual o saldo da minha conta corrente", "consulta_saldo"),
    ("Quero ver quanto tenho na conta", "consulta_saldo"),
    ("Me mostra o extrato e saldo disponivel", "consulta_saldo"),
    ("Quero bloquear meu cartao que foi roubado", "bloquear_cartao"),
    ("Preciso cancelar meu cartao de credito urgente", "bloquear_cartao"),
    ("Meu cartao foi clonado preciso bloquear", "bloquear_cartao"),
    ("Qual a previsao do tempo para amanha", "fora_escopo"),
    ("Me conta uma piada engraçada", "fora_escopo"),
]
stopwords = ["o", "a", "e", "de", "da", "do", "para", "meu", "minha", "uma", "um", "me", "que", "na", "no"]
# 1. PRÉ-PROCESSAMENTO
def pré_processar(texto):
    texto_limpo = re.sub(r'[^\w\s]', '', texto.lower()).strip()
    tokens = texto_limpo.split()

    # TODO 1.1: Filtre as stopwords da lista 'tokens' usando List Comprehension
    tokens_filtrados = [t for t in tokens if t not in stopwords]

    return " ".join(tokens_filtrados)
X_treino_limpo = [pré_processar(item[0]) for item in dados_treino]
y_treino = [item[1] for item in dados_treino]

In [3]:
# 2. VETORIZAÇÃO E TREINAMENTO
vectorizer = TfidfVectorizer()
# TODO 1.2: Aprenda o vocabulário e vetorize 'X_treino_limpo' em um único passo
X_vetorizado = vectorizer.fit_transform(X_treino_limpo)
modelo = LogisticRegression()
modelo.fit(X_vetorizado, y_treino)
# 3. EXTRAÇÃO DE ENTIDADE (REGEX)
def extrair_valor_reais(texto_original):
    match = re.search(r'(\d+)\s*reais', texto_original, re.IGNORECASE)
    # TODO 1.3: Se houver match, converta o grupo 1 para float. Caso contrário, retorne None.
    return float(match.group(1)) if match else None
# 4. PIPELINE NLU
def processar_nlu(mensagem_usuario, threshold=0.55):
    texto_p = pré_processar(mensagem_usuario)
    vetor_input = vectorizer.transform([texto_p])

    probas = modelo.predict_proba(vetor_input)[0]

    # TODO 1.4: Extraia a MAIOR probabilidade (max) e a intenção prevista (argmax em modelo.classes_)
    maior_confianca = max(probas)
    intencao_prevista = modelo.classes_[probas.argmax()]

    # TODO 1.5: Se maior_confianca < threshold ou intencao == "fora_escopo", retorne FALLBACK.
    if maior_confianca < threshold or intencao_prevista == "fora_escopo":
        return {"status": "FALLBACK", "confianca": round(maior_confianca, 4), "mensagem": "Nao entendi sua solicitacao. Transferindo para atendente humano."}
    else:
        valor = extrair_valor_reais(mensagem_usuario)
        return {"status": "SUCESSO", "intencao": intencao_prevista, "confianca": round(maior_confianca, 4), "valor_extraido": valor}

In [5]:
#TESTE
if __name__ == "__main__":
    print("=== LAB 1 - REGRESSAO LOGISTICA BANCARIO ===")
    print()
    print("Teste 1:", processar_nlu("Quero transferir 300 reais para minha mae"))
    print("Teste 2:", processar_nlu("Qual o saldo da minha conta"))
    print("Teste 3:", processar_nlu("Preciso bloquear meu cartao agora"))
    print("Teste 4:", processar_nlu("Qual a temperatura hoje em Curitiba"))
    print()
    print("--- Testes com threshold menor (0.35) ---")
    print("Teste 5:", processar_nlu("Quero transferir 300 reais para minha mae", threshold=0.35))
    print("Teste 6:", processar_nlu("Qual o saldo da minha conta", threshold=0.35))
    print("Teste 7:", processar_nlu("Preciso bloquear meu cartao agora", threshold=0.35))
    print("Teste 8:", processar_nlu("Qual a temperatura hoje em Curitiba", threshold=0.35))

=== LAB 1 - REGRESSAO LOGISTICA BANCARIO ===

Teste 1: {'status': 'FALLBACK', 'confianca': np.float64(0.4191), 'mensagem': 'Nao entendi sua solicitacao. Transferindo para atendente humano.'}
Teste 2: {'status': 'FALLBACK', 'confianca': np.float64(0.437), 'mensagem': 'Nao entendi sua solicitacao. Transferindo para atendente humano.'}
Teste 3: {'status': 'FALLBACK', 'confianca': np.float64(0.4915), 'mensagem': 'Nao entendi sua solicitacao. Transferindo para atendente humano.'}
Teste 4: {'status': 'FALLBACK', 'confianca': np.float64(0.3306), 'mensagem': 'Nao entendi sua solicitacao. Transferindo para atendente humano.'}

--- Testes com threshold menor (0.35) ---
Teste 5: {'status': 'SUCESSO', 'intencao': np.str_('transferencia'), 'confianca': np.float64(0.4191), 'valor_extraido': 300.0}
Teste 6: {'status': 'SUCESSO', 'intencao': np.str_('consulta_saldo'), 'confianca': np.float64(0.437), 'valor_extraido': None}
Teste 7: {'status': 'SUCESSO', 'intencao': np.str_('bloquear_cartao'), 'confian